# 🖱️ AI Virtual Mouse & Gesture Controller

Control your computer's mouse cursor and clicks using real-time hand gestures
captured from a webcam — no physical mouse required.

**Author:** Aditya Pandey ([@Tech4Aditya](https://github.com/Tech4Aditya))
**Built for:** GSSoC 2026 contribution to [ML-CaPsule](https://github.com/Niketkumardheeryan/ML-CaPsule) — closes issue #2017

---

## 📌 Overview

This notebook uses **MediaPipe** to detect 21 hand landmarks in real time,
**OpenCV** to capture and process webcam video, and **PyAutoGUI** to translate
finger positions into system-level cursor movement and click events.

## 🎮 Gesture Guide

| Gesture | Action |
|---|---|
| ☝️ Index finger up, middle finger down | **Move mode** — cursor follows your index fingertip |
| ✌️ Index + middle fingers up, then pinch them together | **Click mode** — registers a left click when distance < threshold |

## ⚠️ Important — Read Before Running

This notebook opens a **live webcam window** (`cv2.imshow`) and controls your
**actual system cursor** via PyAutoGUI. It must be run:
- **Locally** (Jupyter Notebook / JupyterLab on your own machine with a webcam)
- **NOT** on Google Colab or any cloud/headless environment — `cv2.imshow`
  requires a local display and Colab has no webcam access without extra
  JS/WebRTC shims this notebook does not implement
- With **camera and accessibility permissions granted** to your Python
  process/terminal (required on macOS in particular for PyAutoGUI)

Press **`q`** in the video window at any time to stop the loop safely.

## 1. Install Dependencies

Run this once. Restart the kernel after installing if `mediapipe` was not previously installed.

In [1]:
%pip install opencv-python==4.10.0.84 mediapipe==0.10.14 pyautogui==0.9.54 numpy==1.26.4 -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [2]:
import cv2
import time
import math
import numpy as np
import pyautogui
import mediapipe as mp

ImportError: libGL.so.1: cannot open shared object file: No such file or directory

## 3. Hand Detector

Reusable wrapper around MediaPipe Hands that detects hand landmarks in a video frame and exposes convenience methods to fetch landmark positions and finger states.

In [ ]:
class HandDetector:
    """Detects a hand in a BGR frame and tracks 21 landmarks per hand."""

    def __init__(self, mode=False, max_hands=1, detection_confidence=0.7,
                 tracking_confidence=0.7):
        self.mode = mode
        self.max_hands = max_hands
        self.detection_confidence = detection_confidence
        self.tracking_confidence = tracking_confidence

        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=self.mode,
            max_num_hands=self.max_hands,
            min_detection_confidence=self.detection_confidence,
            min_tracking_confidence=self.tracking_confidence,
        )
        self.mp_draw = mp.solutions.drawing_utils

        # Landmark indices for fingertips: thumb, index, middle, ring, pinky
        self.tip_ids = [4, 8, 12, 16, 20]
        self.landmark_list = []
        self.results = None

    def find_hands(self, frame, draw=True):
        """Runs detection on the frame and optionally draws landmarks."""
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        self.results = self.hands.process(frame_rgb)

        if self.results.multi_hand_landmarks:
            for hand_landmarks in self.results.multi_hand_landmarks:
                if draw:
                    self.mp_draw.draw_landmarks(
                        frame, hand_landmarks, self.mp_hands.HAND_CONNECTIONS
                    )
        return frame

    def find_position(self, frame, hand_index=0, draw=True):
        """Returns a list of [id, x, y] for each landmark of the detected hand."""
        self.landmark_list = []

        if self.results and self.results.multi_hand_landmarks:
            if hand_index >= len(self.results.multi_hand_landmarks):
                return self.landmark_list

            hand = self.results.multi_hand_landmarks[hand_index]
            h, w, _ = frame.shape

            for lm_id, lm in enumerate(hand.landmark):
                cx, cy = int(lm.x * w), int(lm.y * h)
                self.landmark_list.append([lm_id, cx, cy])
                if draw:
                    cv2.circle(frame, (cx, cy), 5, (255, 0, 255), cv2.FILLED)

        return self.landmark_list

    def fingers_up(self):
        """Returns a list of 5 booleans indicating which fingers are extended.
        Order: [thumb, index, middle, ring, pinky]."""
        fingers = []
        if not self.landmark_list:
            return [0, 0, 0, 0, 0]

        # Thumb: compare x-coordinates (works for a right hand facing camera)
        if self.landmark_list[self.tip_ids[0]][1] > self.landmark_list[self.tip_ids[0] - 1][1]:
            fingers.append(1)
        else:
            fingers.append(0)

        # Other 4 fingers: tip above the pip joint (lower y = up, image coords)
        for finger_id in range(1, 5):
            tip_y = self.landmark_list[self.tip_ids[finger_id]][2]
            pip_y = self.landmark_list[self.tip_ids[finger_id] - 2][2]
            fingers.append(1 if tip_y < pip_y else 0)

        return fingers

    def find_distance(self, point1, point2, frame, draw=True):
        """Returns euclidean distance between two landmark ids, plus drawing data."""
        x1, y1 = self.landmark_list[point1][1], self.landmark_list[point1][2]
        x2, y2 = self.landmark_list[point2][1], self.landmark_list[point2][2]
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

        if draw:
            cv2.circle(frame, (x1, y1), 10, (255, 0, 255), cv2.FILLED)
            cv2.circle(frame, (x2, y2), 10, (255, 0, 255), cv2.FILLED)
            cv2.line(frame, (x1, y1), (x2, y2), (255, 0, 255), 3)
            cv2.circle(frame, (cx, cy), 10, (255, 0, 255), cv2.FILLED)

        length = math.hypot(x2 - x1, y2 - y1)
        return length, frame, [x1, y1, x2, y2, cx, cy]

## 4. Configuration

Tunable constants:
- `FRAME_REDUCTION` — margin so the cursor can still reach screen edges
- `SMOOTHENING` — higher value = smoother movement but more input lag
- `CLICK_DISTANCE_THRESHOLD` — pixel distance between thumb & index to count as a pinch
- `CLICK_COOLDOWN` — minimum seconds between two clicks (prevents double-firing)

In [ ]:
CAM_WIDTH, CAM_HEIGHT = 640, 480
FRAME_REDUCTION = 100
SMOOTHENING = 5
CLICK_DISTANCE_THRESHOLD = 35
CLICK_COOLDOWN = 0.4

## 5. Main Loop

Run the cell below to start the webcam and begin tracking. A window titled **"AI Virtual Mouse"** will open. Press **`q`** in that window to stop.

In [ ]:
def run_virtual_mouse():
    prev_x, prev_y = 0, 0
    curr_x, curr_y = 0, 0
    last_click_time = 0

    screen_width, screen_height = pyautogui.size()
    pyautogui.FAILSAFE = False  # allow cursor near screen corners without aborting

    cap = cv2.VideoCapture(0)
    cap.set(3, CAM_WIDTH)
    cap.set(4, CAM_HEIGHT)

    detector = HandDetector(max_hands=1, detection_confidence=0.8)
    prev_time = 0

    print("AI Virtual Mouse started. Press 'q' in the video window to quit.")

    while True:
        success, frame = cap.read()
        if not success:
            print("Failed to grab frame from webcam.")
            break

        frame = cv2.flip(frame, 1)  # mirror for natural interaction
        frame = detector.find_hands(frame)
        landmarks = detector.find_position(frame, draw=False)

        cv2.rectangle(
            frame,
            (FRAME_REDUCTION, FRAME_REDUCTION),
            (CAM_WIDTH - FRAME_REDUCTION, CAM_HEIGHT - FRAME_REDUCTION),
            (255, 0, 255), 2,
        )

        if landmarks:
            index_x, index_y = landmarks[8][1], landmarks[8][2]
            fingers = detector.fingers_up()

            # ---- MOVE MODE: only index finger extended ----
            if fingers[1] == 1 and fingers[2] == 0:
                mapped_x = np.interp(
                    index_x, (FRAME_REDUCTION, CAM_WIDTH - FRAME_REDUCTION), (0, screen_width)
                )
                mapped_y = np.interp(
                    index_y, (FRAME_REDUCTION, CAM_HEIGHT - FRAME_REDUCTION), (0, screen_height)
                )

                curr_x = prev_x + (mapped_x - prev_x) / SMOOTHENING
                curr_y = prev_y + (mapped_y - prev_y) / SMOOTHENING

                pyautogui.moveTo(screen_width - curr_x, curr_y)
                cv2.circle(frame, (index_x, index_y), 12, (255, 0, 255), cv2.FILLED)
                prev_x, prev_y = curr_x, curr_y

            # ---- CLICK MODE: index + middle extended, check pinch distance ----
            if fingers[1] == 1 and fingers[2] == 1:
                length, frame, line_info = detector.find_distance(8, 12, frame)

                if length < CLICK_DISTANCE_THRESHOLD:
                    now = time.time()
                    if now - last_click_time > CLICK_COOLDOWN:
                        cv2.circle(frame, (line_info[4], line_info[5]), 12, (0, 255, 0), cv2.FILLED)
                        pyautogui.click()
                        last_click_time = now

        # ---- FPS overlay ----
        curr_time = time.time()
        fps = 1 / (curr_time - prev_time) if prev_time else 0
        prev_time = curr_time
        cv2.putText(frame, f"FPS: {int(fps)}", (20, 40),
                    cv2.FONT_HERSHEY_PLAIN, 2, (0, 255, 0), 2)

        cv2.imshow("AI Virtual Mouse", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

run_virtual_mouse()

## ⚙️ Known Limitations

- Tuned for a right hand facing the camera; left-hand thumb detection may need
  the `fingers_up()` comparison flipped, or handedness detection via
  `results.multi_handedness`.
- Performance depends on webcam quality and lighting conditions.
- Single-hand tracking only (`max_hands=1`) by default for stability.
- Must be run in a local Jupyter environment with display + webcam access —
  will not work on Colab or headless servers as-is.

## 🛑 Emergency Stop

If the video window becomes unresponsive or the cursor behaves unexpectedly,
move your mouse to a screen corner manually to regain control, or interrupt
the kernel (⏹ button / `Kernel → Interrupt`) — `cap.release()` and
`cv2.destroyAllWindows()` will not run automatically on interrupt, so also
run the cell below afterward to clean up.

In [ ]:
# Run this if you interrupted the kernel and the webcam LED is still on
# or the video window is stuck open.
try:
    cap.release()
except NameError:
    pass
cv2.destroyAllWindows()
print("Cleaned up.")